# Libraries

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from dhydro_utils import run_simulations_mesh, create_raw_dataset_folder, create_mesh_dhydro
from graph_creation import Mesh, plot_faces, save_polygon_to_file, find_closest_nodes

# Run simulations

## Run only this path (new catchment)
Run these code cells in order:
1) Cell 2 (imports)
2) Cell 4 (single simulation from shapefile + DEM + hydrograph)
3) Cell 19 (checks + dataset folders)
4) Cell 20 (build and save train/test pickles)

In [ ]:
import os
import numpy as np

# 1) Path to an EXISTING D-Hydro model template (contains FlowFM.mdu, run.bat, etc.)
model_folder = r'C:\path\to\your\dflowfm' #######

# 2) Output raw dataset folder inside this repo's database folder
save_folder = 'raw_datasets_new_catchment'  #######
create_raw_dataset_folder(save_folder)

# 3) Catchment boundary from shapefile -> convert to .pol for D-Hydro
#    Use the first feature in the shapefile.
shapefile_path = r'C:\path\to\your\catchment_boundary.shp'   #######
polygon_file = os.path.join(save_folder, 'catchment_boundary.pol')

if not os.path.exists(shapefile_path):
    raise FileNotFoundError(f'Shapefile not found: {shapefile_path}')

try:
    import fiona
    from shapely.geometry import shape as geom_shape

    with fiona.open(shapefile_path, 'r') as shp:
        feat = shp[0]
        geom = geom_shape(feat['geometry'])

    if geom.geom_type == 'MultiPolygon':
        geom = max(geom.geoms, key=lambda g: g.area)

    # Optional simplification to reduce very dense boundaries.
    # Set to 0 to disable simplification.
    simplify_tolerance = 0
    if simplify_tolerance > 0:
        geom = geom.simplify(tolerance=simplify_tolerance, preserve_topology=True)

    save_polygon_to_file(geom, polygon_file)
    print(f'Created polygon file: {polygon_file}')
except Exception as e:
    raise RuntimeError('Could not convert shapefile boundary to .pol') from e

# 4) DEM input
#    Preferred: provide an XYZ file directly (x y z columns)
#DEM_xyz_file = r'C:\path\to\your\DEM.xyz'

#    Optional: if you only have .tif and rasterio is available, convert once to XYZ
DEM_tif_file = r'C:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\your_dem.tif' #######
if (not os.path.exists(DEM_xyz_file)) and os.path.exists(DEM_tif_file):
    try:
        import rasterio

        with rasterio.open(DEM_tif_file) as src:
            arr = src.read(1)
            rows, cols = np.where(~np.isnan(arr))
            xs, ys = src.xy(rows, cols)
            zs = arr[rows, cols]
            xyz = np.column_stack([np.array(xs), np.array(ys), zs])

        DEM_xyz_file = os.path.join(save_folder, 'DEM_from_tif.xyz')
        np.savetxt(DEM_xyz_file, xyz, fmt='%.3f %.3f %.5f')
        print(f'Created DEM XYZ: {DEM_xyz_file}')
    except Exception as e:
        raise RuntimeError(
            'Could not convert DEM .tif to .xyz automatically. '
            'Create DEM.xyz manually (x y z columns) and set DEM_xyz_file to that path.'
        ) from e

# 5) Hydrograph input txt: two columns [time_min, discharge_m3s]
hydrograph_txt = r'C:\path\to\your\hydrograph_1.txt'  #######
h = np.loadtxt(hydrograph_txt)
assert h.ndim == 2 and h.shape[1] >= 2, 'Hydrograph must have at least 2 columns: time[min], discharge[m3/s]'

# Convert minutes to seconds for D-Hydro boundary condition writer
hydrograph = (h[:, 0] * 60.0, h[:, 1])

# 6) Run exactly 1 simulation
simulation_id = 1
number_of_multiscales = 4

df = run_simulations_mesh(
    n_sim=1,
    model_folder=model_folder,
    save_folder=save_folder,
    start_sim=simulation_id,
    DEM_file=DEM_xyz_file,
    polygon_file=polygon_file,
    breach_coords=None,
    number_of_multiscales=number_of_multiscales,
    random_hydrograph=False,
    hydrograph=hydrograph,
)

df

## New catchment (single simulation)
Use this section to run 1 simulation for your catchment and generate the required raw dataset files.

Expected outputs in `save_folder`:
- `Simulations/output_<id>_map.nc`
- `DEM/DEM_<id>.xyz`
- `Hydrograph/Hydrograph_<id>.txt`
- `Geometry/polygon_<id>.pol`
- `overview.csv`

## Dike ring

In [ ]:
# model_folder = 'C:\\Users\\rbentivoglio\\Documents\\PhD\\My PhD\\D-HYDRO\\dijkring_15.dsproj_data\\FlowFM\\input\\dflowfm'
# save_folder = 'raw_datasets_dk15'
# polygon_file = 'dijkring_15.pol'
# DEM_file = model_folder+'/DEM_25m_15.xyz'

In [ ]:
# from shapely.geometry import Polygon
# from shapely import geometry
# import fiona
# import numpy as np

# shapefile_path = "C:/Users/rbentivoglio/Documents/PhD/GeoData/Nederland/dijkring_15.shp"

# with fiona.open(shapefile_path, "r") as shapefile:
#   shape = shapefile[0]

#   line = geometry.LineString(shape['geometry'].coordinates[0])
#   polygon = geometry.Polygon(line)

# polygon = Polygon(np.array(shape.geometry.coordinates)[0])
# polygon = polygon.simplify(tolerance=300, preserve_topology=True)

# save_polygon_to_file(polygon, polygon_file)

In [ ]:
# plt.plot(np.array(polygon.boundary.xy)[0], np.array(polygon.boundary.xy)[1])

# # project polygon boundary to a line
# line = geometry.LineString(polygon.boundary.coords)

# # uniform points along the line
# distance = 8700 #6500
# points = [line.interpolate(i * distance + 2000).xy for i in range(int(line.length / distance))]
# points = np.array(points).squeeze()

# # plot points
# print("Number of points: ", len(points))
# plt.plot(points[:,0], points[:,1], 'x');

In [ ]:
# mesh = create_mesh_dhydro(polygon_file, 4)

# boundary_edges = np.where((mesh.edge_faces.reshape(-1,2) == -1).sum(1) == 1)[0]
# boundary_nodes = mesh.mesh_nodes[mesh.edge_nodes.reshape(-1,2)[boundary_edges]].reshape(-1,2)

In [ ]:
# np.random.seed(42)

# total_time = 3600*24*4
# time_resolution = 3600
# min_discharge = 0
# peak_value = (np.random.rand(len(points)))*250+750
# Ln = 10
# Q  = 0.3

# time_steps = int(total_time/time_resolution)+1
# time_x = np.linspace(0.25, 0.8, time_steps)
# time_hydrograph = time_x - time_x.min()
# time_hydrograph = time_hydrograph/time_hydrograph.max() * total_time

# for i, point in enumerate(points):
#     boundary_edge = find_closest_nodes(boundary_nodes, point, top_n=3)
#     coords = boundary_nodes[boundary_edge]

#     F = time_x**2
#     y = F * (Ln - 1) / np.sqrt((Ln * F - 1)**2 + F * (F - 1)**2 * (Ln - 1)**2 * Q**2)
#     y = y/y.max() * (peak_value[i]-min_discharge) + min_discharge

#     hydrograph = (time_hydrograph, y)
#     df = run_simulations_mesh(1, model_folder, save_folder, start_sim=101+i, 
#                             DEM_file=DEM_file, polygon_file=polygon_file, 
#                             breach_coords=coords[1:], number_of_multiscales=4,
#                             random_hydrograph=False, hydrograph=hydrograph)

# df

## Synthetic dataset

In [ ]:
# model_folder = 'C:\\Users\\rbentivoglio\\Documents\\PhD\\My PhD\\D-HYDRO\\SWEGNN_mesh.dsproj_data\\FlowFM\\input\\dflowfm'
# save_folder = 'raw_datasets_mesh'

# create_raw_dataset_folder(save_folder)

In [ ]:
# df = run_simulations_mesh(1, model_folder, save_folder, start_sim=91, 
#                             # DEM_file=DEM_file, polygon_file=polygon_file, breach_coords=coords[1:],
#                             noise_octave=(1,5), DEM_multiplier=(0.5,5), slope_multiplier=(0,0.005), 
#                             num_vertices_polygon=(24,28), number_of_multiscales=4, ellipticality=(1,2), grid_size=83,
#                             random_hydrograph=True, min_discharge=0, peak=(150,300), shape=(0,2))

# Visualize results

In [ ]:
# mesh = Mesh()
# _id = 106
# nc_file = f'{save_folder}/Simulations/output_{_id}_map.nc'
# mesh._import_from_map_netcdf(nc_file)
# mesh._get_derived_attributes()
# mesh._import_DEM(f"{save_folder}\\DEM\\DEM_{_id}.xyz")
# mesh

In [ ]:
# nc_dataset = xr.open_dataset(nc_file)

# waterdepth = nc_dataset['mesh2d_waterdepth'].data

# fig, axs = plt.subplots(1, 2, figsize=(12,5))
# plot_faces(mesh, ax=axs[1], face_value=waterdepth[:,-1], cmap='Blues', 
#            edgecolor='k', linewidths=0.1,
#            )

# cbar = plt.colorbar(axs[1].collections[0], ax=axs[1], orientation='vertical')
# # cbar.set_label('Color')

# plot_faces(mesh, ax=axs[0], face_value=mesh.DEM, cmap='terrain', 
#            edgecolor='k', linewidths=0.1,
#            )
# cbar = plt.colorbar(axs[0].collections[0], ax=axs[0], orientation='vertical')

## Convert 1 raw simulation to dataset pickles (smoke test)
Run these cells after the simulation cell has produced files in `raw_datasets_new_catchment`.

In [ ]:
import os
from graph_creation import create_mesh_dataset, save_database, create_dataset_folders

# Inputs
raw_folder = 'raw_datasets_new_catchment'   # folder generated by run_simulations_mesh  #######
catchment_name = 'your_catchment_name'      # output pickle name  ########
simulation_id = 1                            # simulation id used above
with_multiscale = True                       # keep True if using MSGNN checkpoints

# Quick existence checks
required_files = [
    f'{raw_folder}/Simulations/output_{simulation_id}_map.nc',
    f'{raw_folder}/DEM/DEM_{simulation_id}.xyz',
    f'{raw_folder}/Hydrograph/Hydrograph_{simulation_id}.txt',
]
missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing expected raw files:\n' + '\n'.join(missing))

# Create output folder structure for model datasets
create_dataset_folders(dataset_folder='datasets')

In [ ]:
# Build dataset from exactly one simulation
mesh_dataset = create_mesh_dataset(
    dataset_folder=raw_folder,
    n_sim=1,
    start_sim=simulation_id,
    with_multiscale=with_multiscale,
    number_of_multiscales=4,
)

# Save the same single sample to both train and test for a smoke test
# (use more simulations later for proper evaluation)
save_database(mesh_dataset, name=catchment_name, out_path='datasets/train') ########
save_database(mesh_dataset, name=catchment_name, out_path='datasets/test') ########

print('Created:')
print(f'datasets/train/{catchment_name}.pkl') #########
print(f'datasets/test/{catchment_name}.pkl') #########